In [ ]:
# In[1]:

import pandas as pd

# 1) Read CSVs and parse timestamps with UTC
df_metric = pd.read_csv("metric.csv")
df_trace = pd.read_csv("trace.csv")
df_log = pd.read_csv("log.csv")
df_error = pd.read_csv("error_logs.csv")

for df in (df_metric, df_trace, df_log, df_error):
    if "timestamp" in df.columns:
        df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s", utc=True)

# 2) Prepare unique (source, component, kpi_name) lists (up to 50 rows each)
metric_unique = (
    df_metric[["cmdb_id", "kpi_name"]]
    .drop_duplicates()
    .rename(columns={"cmdb_id": "component"})
)
metric_unique.insert(0, "source", "metric")

trace_unique = (
    df_trace[["cmdb_id", "trace_name"]]
    .drop_duplicates()
    .rename(columns={"cmdb_id": "component", "trace_name": "kpi_name"})
)
trace_unique.insert(0, "source", "trace")

log_unique = (
    df_log[["cmdb_id", "log_name"]]
    .drop_duplicates()
    .rename(columns={"cmdb_id": "component", "log_name": "kpi_name"})
)
log_unique.insert(0, "source", "log")

metric_unique = metric_unique[["source", "component", "kpi_name"]].head(50)
trace_unique = trace_unique[["source", "component", "kpi_name"]].head(50)
log_unique = log_unique[["source", "component", "kpi_name"]].head(50)

# 3) For each component-KPI time series compute count and global P95 (full file)
# Metric
metric_stats = (
    df_metric.groupby(["cmdb_id", "kpi_name"], dropna=False)["value"]
    .agg(count="count", p95=lambda s: s.quantile(0.95))
    .reset_index()
    .rename(columns={"cmdb_id": "component"})
)
metric_stats.insert(0, "source", "metric")

# Trace
trace_stats = (
    df_trace.groupby(["cmdb_id", "trace_name"], dropna=False)["value"]
    .agg(count="count", p95=lambda s: s.quantile(0.95))
    .reset_index()
    .rename(columns={"cmdb_id": "component", "trace_name": "kpi_name"})
)
trace_stats.insert(0, "source", "trace")

# Log
log_stats = (
    df_log.groupby(["cmdb_id", "log_name"], dropna=False)["value"]
    .agg(count="count", p95=lambda s: s.quantile(0.95))
    .reset_index()
    .rename(columns={"cmdb_id": "component", "log_name": "kpi_name"})
)
log_stats.insert(0, "source", "log")

# Combine stats and select top 20 by count
combined_stats = pd.concat([metric_stats, trace_stats, log_stats], ignore_index=True, sort=False)
combined_stats_top20 = combined_stats.sort_values("count", ascending=False).reset_index(drop=True).head(20)
# Keep only required columns
combined_stats_top20 = combined_stats_top20[["source", "component", "kpi_name", "count", "p95"]]

# 4) error_logs: distinct cmdb_id and number of error log rows per cmdb_id (top 20)
error_counts = (
    df_error.groupby("cmdb_id", dropna=False)
    .size()
    .reset_index(name="error_count")
    .rename(columns={"cmdb_id": "component"})
    .sort_values("error_count", ascending=False)
    .reset_index(drop=True)
)
error_counts_top20 = error_counts.head(20)

# Final displayed outputs (compact)
metric_unique, trace_unique, log_unique, combined_stats_top20, error_counts_top20

```
Out[1]:
```
Summary of results (plain English):

- Unique component-KPI pairs (samples):
  - Metric: many components were observed (examples shown include ts-admin-basic-info-service, ts-admin-order-service, ts-admin-route-service, ts-admin-travel-service, ts-auth-service). Up to 50 metric component–kpi pairs were listed.
  - Trace: trace entries are mostly under component "root" with many trace.* KPIs (e.g., trace.to_ts-admin-basic-info-service.duration_p95, error_rate, row_count). Up to 50 trace rows were listed.
  - Log: many components with log.error_count and log.row_count (examples include ts-admin-basic-info-service, ts-admin-travel-service, ts-auth-mongo, ts-config-mongo). Up to 50 log rows were listed.

- Top time-series statistics (top 20 by count across metric/trace/log):
  - The top entries all have count = 25 (i.e., 25 data points each).
  - Highest-count / highest p95 examples:
    - log | ts-ui-dashboard | log.row_count — count 25, p95 = 305.0
    - metric | ts-admin-basic-info-service | cpu — count 25, p95 ≈ 1.7006
    - metric | ts-admin-basic-info-service | diskio — count 25, p95 ≈ 59018.14
    - metric | ts-admin-basic-info-service | mem — count 25, p95 ≈ 2.30025e+08
    - metric | ts-admin-basic-info-service | socket — count 25, p95 ≈ 7.0533
    - Several other admin-* services (order, route, travel) show count 25 with their respective p95 values.

- error_logs.csv:
  - No error log rows were found per the grouping: the resulting table is empty (no cmdb_id with error counts).

Conclusion:
- All metric/trace/log series used the full available series (most series have 25 points).
- The highest observed p95 in the top table is for log.row_count on ts-ui-dashboard (p95 = 305), and metric p95 values for ts-admin-basic-info-service indicate high memory (≈2.3e8) and large diskio values (~59018) in their 95th percentiles.
- There are no error log rows detected in error_logs.csv.

The original code execution output of IPython Kernel is also provided below for reference:

(    source                    component    kpi_name
0   metric  ts-admin-basic-info-service         cpu
1   metric  ts-admin-basic-info-service      diskio
2   metric  ts-admin-basic-info-service  latency-50
3   metric  ts-admin-basic-info-service  latency-90
4   metric  ts-admin-basic-info-service         mem
..     ...                          ...         ...
45  metric              ts-auth-service  latency-50
46  metric              ts-auth-service  latency-90
47  metric              ts-auth-service         mem
48  metric              ts-auth-service      socket
49  metric              ts-auth-service    workload

[50 rows x 3 columns],    source component                                           kpi_name
0   trace      root  trace.to_ts-admin-basic-info-service.duration_...
1   trace      root  trace.to_ts-admin-basic-info-service.duration_p95
2   trace      root    trace.to_ts-admin-basic-info-service.error_rate
3   trace      root     trace.to_ts-admin-basic-info-service.row_count
4   trace      root     trace.to_ts-admin-travel-service.duration_mean
..    ...       ...                                                ...
45  trace      root          trace.to_ts-preserve-service.duration_p95
46  trace      root            trace.to_ts-preserve-service.error_rate
47  trace      root             trace.to_ts-preserve-service.row_count
48  trace      root             trace.to_ts-seat-service.duration_mean
49  trace      root              trace.to_ts-seat-service.duration_p95

[50 rows x 3 columns],    source                    component         kpi_name
0     log  ts-admin-basic-info-service  log.error_count
1     log  ts-admin-basic-info-service    log.row_count
2     log      ts-admin-travel-service  log.error_count
3     log      ts-admin-travel-service    log.row_count
4     log         ts-assurance-service  log.error_count
..    ...                          ...              ...
51    log                ts-auth-mongo    log.row_count
56    log              ts-config-mongo  log.error_count
57    log              ts-config-mongo    log.row_count
58    log             ts-consign-mongo  log.error_count
59    log             ts-consign-mongo    log.row_count

[50 rows x 3 columns],     source                    component         kpi_name  count           p95
0      log              ts-ui-dashboard    log.row_count     25  3.050000e+02
1   metric  ts-admin-basic-info-service              cpu     25  1.700573e+00
2   metric  ts-admin-basic-info-service           diskio     25  5.901814e+04
3   metric  ts-admin-basic-info-service       latency-50     25  1.753028e-02
4   metric  ts-admin-basic-info-service       latency-90     25  2.405504e-02
5   metric  ts-admin-basic-info-service              mem     25  2.300251e+08
6   metric  ts-admin-basic-info-service           socket     25  7.053333e+00
7   metric  ts-admin-basic-info-service         workload     25  6.720867e-01
8   metric       ts-admin-order-service              cpu     25  1.050281e-01
9   metric       ts-admin-order-service           diskio     25  4.554680e+04
10  metric       ts-admin-order-service              mem     25  1.948775e+08
11  metric       ts-admin-order-service           socket     25  3.516667e+00
12  metric       ts-admin-route-service              cpu     25  1.104244e-01
13  metric       ts-admin-route-service           diskio     25  4.980810e+04
14  metric       ts-admin-route-service              mem     25  1.926468e+08
15  metric       ts-admin-route-service           socket     25  3.616667e+00
16  metric      ts-admin-travel-service              cpu     25  4.890521e-01
17  metric      ts-admin-travel-service           diskio     25  5.896230e+04
18     log       ts-order-other-service  log.error_count     25  0.000000e+00
19     log       ts-order-other-service    log.row_count     25  5.074000e+02, Empty DataFrame
Columns: [component, error_count]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# Reuse existing DataFrames if present; otherwise read files
try:
    df_metric
    df_trace
    df_log
    df_error
except NameError:
    df_metric = pd.read_csv("metric.csv")
    df_trace = pd.read_csv("trace.csv")
    df_log = pd.read_csv("log.csv")
    df_error = pd.read_csv("error_logs.csv")

# Ensure timestamps parsed as UTC datetimes
for df in (df_metric, df_trace, df_log, df_error):
    if "timestamp" in df.columns:
        df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s", utc=True)

# Candidate components
candidates = ["ts-auth-service", "ts-order-service", "ts-route-service", "ts-train-service", "ts-travel-service"]

# Helper to compute global stats (count, p95, p05) for a dataframe given component and kpi columns
def compute_stats(df, comp_col, kpi_col):
    stats = (
        df.groupby([comp_col, kpi_col], dropna=False)["value"]
        .agg(count="count", p95=lambda s: s.quantile(0.95), p05=lambda s: s.quantile(0.05))
        .reset_index()
        .rename(columns={comp_col: "component", kpi_col: "kpi_name"})
    )
    return stats

# 1) Compute global stats for candidate components for each source (full series)
metric_stats_all = compute_stats(df_metric, "cmdb_id", "kpi_name")
metric_stats = metric_stats_all[metric_stats_all["component"].isin(candidates)].copy()
metric_stats.insert(0, "source", "metric")
metric_stats = metric_stats[["source", "component", "kpi_name", "count", "p95", "p05"]]

trace_stats_all = compute_stats(df_trace, "cmdb_id", "trace_name")
trace_stats = trace_stats_all[trace_stats_all["component"].isin(candidates)].copy()
trace_stats.insert(0, "source", "trace")
trace_stats = trace_stats[["source", "component", "kpi_name", "count", "p95", "p05"]]

log_stats_all = compute_stats(df_log, "cmdb_id", "log_name")
log_stats = log_stats_all[log_stats_all["component"].isin(candidates)].copy()
log_stats.insert(0, "source", "log")
log_stats = log_stats[["source", "component", "kpi_name", "count", "p95", "p05"]]

# Combine for output (limit top 50 rows)
combined_stats_candidates = pd.concat([metric_stats, trace_stats, log_stats], ignore_index=True, sort=False)
combined_stats_candidates = combined_stats_candidates.sort_values(["component", "source", "kpi_name"]).reset_index(drop=True)
combined_stats_candidates_head50 = combined_stats_candidates.head(50)

# 2) Incident window filtering
start = pd.to_datetime("2024-01-25 03:14:30", utc=True)
end = pd.to_datetime("2024-01-25 03:44:30", utc=True)

# Heuristics for KPI directionality
low_patterns = ["success", "success_rate", "availability"]  # low-sided (decrease is bad)
error_patterns = ["error", "error_rate", "fail", "failure", "drop"]  # error-like (increase is bad)
# For row_count and throughput, treat as high-sided (increase is anomaly) per instruction
# Default: high-sided (value > p95 is anomaly)

def is_low_sided(kname):
    ln = str(kname).lower()
    return any(p in ln for p in low_patterns)

def is_error_like(kname):
    ln = str(kname).lower()
    return any(p in ln for p in error_patterns)

# Function to detect anomalies for a given telemetry DF, its column names, and global stats
def detect_anomalies(df, comp_col, kpi_col, source_name, stats_df):
    # Prepare working df: filter candidate components and time window
    df_work = df[df[comp_col].isin(candidates)].copy()
    df_work = df_work[(df_work["timestamp"] >= start) & (df_work["timestamp"] <= end)]
    if df_work.empty:
        return pd.DataFrame(columns=["source","component","kpi_name","timestamp","value","p95","p05","anomaly_type","exceedance_ratio"])
    # Normalize column names for merging
    df_work = df_work.rename(columns={comp_col: "component", kpi_col: "kpi_name"})
    # Merge global stats (precomputed p95/p05) - left join to keep rows that match a known KPI
    merged = df_work.merge(stats_df[["component","kpi_name","p95","p05"]], on=["component","kpi_name"], how="left")
    # Drop rows without global thresholds (no full-series data)
    merged = merged.dropna(subset=["p95","p05"], how="any")
    if merged.empty:
        return pd.DataFrame(columns=["source","component","kpi_name","timestamp","value","p95","p05","anomaly_type","exceedance_ratio"])
    # Determine anomalies
    def evaluate_row(row):
        k = row["kpi_name"]
        val = row["value"]
        p95 = row["p95"]
        p05 = row["p05"]
        ln = str(k).lower()
        # If KPI is low-sided (e.g., success rate), detect value < p05
        if is_low_sided(k):
            if pd.notna(p05) and val < p05:
                return ("low", val / p05 if p05 != 0 else np.inf)
            else:
                return (None, None)
        else:
            # High-sided default. For error-like KPIs, also treat as high-sided (increases)
            if pd.notna(p95) and val > p95:
                return ("high", val / p95 if p95 != 0 else np.inf)
            else:
                return (None, None)
    evals = merged.apply(lambda r: evaluate_row(r), axis=1)
    merged[["anomaly_type","exceedance_ratio"]] = pd.DataFrame(evals.tolist(), index=merged.index)
    anomalies = merged[merged["anomaly_type"].notna()].copy()
    if anomalies.empty:
        return pd.DataFrame(columns=["source","component","kpi_name","timestamp","value","p95","p05","anomaly_type","exceedance_ratio"])
    # Prepare final columns and formatting
    anomalies["source"] = source_name
    anomalies["timestamp"] = anomalies["timestamp"].dt.tz_convert("UTC").dt.strftime("%Y-%m-%dT%H:%M:%SZ")
    anomalies = anomalies[["source","component","kpi_name","timestamp","value","p95","p05","anomaly_type","exceedance_ratio"]]
    return anomalies

# Detect anomalies per source using the precomputed full-series stats
anom_metric = detect_anomalies(df_metric, "cmdb_id", "kpi_name", "metric", metric_stats)
anom_trace = detect_anomalies(df_trace, "cmdb_id", "trace_name", "trace", trace_stats)
anom_log = detect_anomalies(df_log, "cmdb_id", "log_name", "log", log_stats)

# Combine anomalies and sort by exceedance_ratio desc, limit top 50
anomalies_all = pd.concat([anom_metric, anom_trace, anom_log], ignore_index=True, sort=False)
if not anomalies_all.empty:
    anomalies_all = anomalies_all.sort_values("exceedance_ratio", ascending=False).reset_index(drop=True)
anomalies_top50 = anomalies_all.head(50)

# 4) Summary aggregated by component for listed candidates
summary_rows = []
# Determine components that have any KPI rows in the files (global stats combined)
present_components = set(combined_stats_candidates["component"].unique())

for comp in candidates:
    if comp not in present_components:
        summary_rows.append({"component": comp, "total_anomalies_count": "no data", "earliest_anomaly_timestamp": "no data", "max_exceedance_ratio": "no data"})
    else:
        comp_anoms = anomalies_all[anomalies_all["component"] == comp]
        if comp_anoms.empty:
            summary_rows.append({"component": comp, "total_anomalies_count": 0, "earliest_anomaly_timestamp": pd.NaT, "max_exceedance_ratio": np.nan})
        else:
            # earliest timestamp (already ISO string); convert back to datetime for min
            earliest = pd.to_datetime(comp_anoms["timestamp"], utc=True)
            max_exc = comp_anoms["exceedance_ratio"].max()
            summary_rows.append({"component": comp, "total_anomalies_count": int(comp_anoms.shape[0]), "earliest_anomaly_timestamp": earliest.min().strftime("%Y-%m-%dT%H:%M:%SZ"), "max_exceedance_ratio": float(max_exc)})

summary_df = pd.DataFrame(summary_rows, columns=["component","total_anomalies_count","earliest_anomaly_timestamp","max_exceedance_ratio"])

# Outputs (compact): combined_stats_candidates_head50, anomalies_top50, summary_df
combined_stats_candidates_head50, anomalies_top50, summary_df

```
Out[2]:
```
Summary of findings (plain English):

Key anomaly highlights
- The most severe single anomaly is in ts-auth-service (trace.to_ts-auth-service.duration_p95 and trace.from_ts-auth-service.duration_p95) at 2024-01-25T03:18:00Z: value 0.066443 vs global p95 0.003220 → exceedance ratio ~20.63×. This is a large latency spike.
- Many anomalies are high-sided (spikes): mostly trace duration_p95/duration_mean and some metric latency-90 / row_count increases. No clear low-sided (drop/success-rate) anomalies were detected in the candidate set.

Per-component compact summary (candidate components only)
- ts-auth-service: total anomalies = 34; earliest anomaly = 2024-01-25T03:17:00Z; max exceedance ratio = 20.6339.
- ts-order-service: total anomalies = 69; earliest anomaly = 2024-01-25T03:17:00Z; max exceedance ratio = 2.7878.
- ts-route-service: total anomalies = 46; earliest anomaly = 2024-01-25T03:17:00Z; max exceedance ratio = 1.4150.
- ts-train-service: total anomalies = 44; earliest anomaly = 2024-01-25T03:17:00Z; max exceedance ratio = 1.5918.
- ts-travel-service: total anomalies = 89 (highest); earliest anomaly = 2024-01-25T03:17:00Z; max exceedance ratio = 1.9627.

Additional notes
- Anomalies are concentrated in trace KPIs (duration_p95, duration_mean, row_count) and metric latency KPIs during the incident window (2024-01-25 03:14:30 to 03:44:30 UTC).
- ts-travel-service produced the largest number of anomalies overall, but ts-auth-service contains the largest single severity spike (20× p95).
- Based on severity, investigate ts-auth-service latency at ~03:17–03:18 UTC first; also examine ts-travel-service and ts-order-service for broader elevated activity during the same window.

The original code execution output of IPython Kernel is also provided below for reference:

(    source         component                                   kpi_name  count           p95           p05
0      log   ts-auth-service                            log.error_count     25      0.000000      0.000000
1      log   ts-auth-service                              log.row_count     25    359.200000    188.000000
2   metric   ts-auth-service                                        cpu     25     16.245113     11.122586
3   metric   ts-auth-service                                     diskio     25  55708.242897  44929.703462
4   metric   ts-auth-service                                 latency-50     25      0.279313      0.206919
..     ...               ...                                        ...    ...           ...           ...
45   trace  ts-order-service  trace.from_ts-preserve-service.error_rate     23      0.000000      0.000000
46   trace  ts-order-service   trace.from_ts-preserve-service.row_count     23      6.000000      1.200000
47   trace  ts-order-service   trace.from_ts-seat-service.duration_mean     25      0.008271      0.004910
48   trace  ts-order-service    trace.from_ts-seat-service.duration_p95     25      0.018641      0.010666
49   trace  ts-order-service      trace.from_ts-seat-service.error_rate     25      0.000000      0.000000

[50 rows x 6 columns],     source          component                                           kpi_name             timestamp       value         p95         p05 anomaly_type  exceedance_ratio
0    trace    ts-auth-service              trace.to_ts-auth-service.duration_p95  2024-01-25T03:18:00Z    0.066443    0.003220    0.001777         high         20.633932
1    trace    ts-auth-service            trace.from_ts-auth-service.duration_p95  2024-01-25T03:18:00Z    0.066443    0.003220    0.001777         high         20.633932
2    trace   ts-order-service            trace.from_ts-seat-service.duration_p95  2024-01-25T03:37:00Z    0.051968    0.018641    0.010666         high          2.787845
3    trace   ts-order-service        trace.from_ts-preserve-service.duration_p95  2024-01-25T03:37:00Z    0.070795    0.027673    0.011666         high          2.558260
4    trace   ts-order-service  trace.from_ts-inside-payment-service.duration_p95  2024-01-25T03:37:00Z    0.057626    0.028890    0.003540         high          1.994681
..     ...                ...                                                ...                   ...         ...         ...         ...          ...               ...
45   trace  ts-travel-service                trace.to_ts-train-service.row_count  2024-01-25T03:18:00Z  282.000000  251.200000  136.200000         high          1.122611
46  metric  ts-travel-service                                         latency-90  2024-01-25T03:41:00Z    5.639460    5.029179    0.080940         high          1.121348
47   trace  ts-travel-service       trace.to_ts-ticketinfo-service.duration_mean  2024-01-25T03:25:00Z    0.030487    0.027200    0.018159         high          1.120845
48   trace   ts-order-service  trace.from_ts-inside-payment-service.duration_...  2024-01-25T03:31:00Z    0.010118    0.009081    0.003351         high          1.114158
49   trace   ts-train-service         trace.from_ts-travel2-service.duration_p95  2024-01-25T03:41:00Z    0.005188    0.004675    0.004169         high          1.109692

[50 rows x 9 columns],            component  total_anomalies_count earliest_anomaly_timestamp  max_exceedance_ratio
0    ts-auth-service                     34       2024-01-25T03:17:00Z             20.633932
1   ts-order-service                     69       2024-01-25T03:17:00Z              2.787845
2   ts-route-service                     46       2024-01-25T03:17:00Z              1.414953
3   ts-train-service                     44       2024-01-25T03:17:00Z              1.591778
4  ts-travel-service                     89       2024-01-25T03:17:00Z              1.962685)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import numpy as np

# Ensure trace dataframe and timestamps are parsed (reuse df_trace)
df_trace["timestamp"] = pd.to_datetime(df_trace["timestamp"], unit="s", utc=True)

# Incident window
start = pd.to_datetime("2024-01-25 03:14:30", utc=True)
end = pd.to_datetime("2024-01-25 03:44:30", utc=True)

# 1) Identify trace_names related to ts-auth-service (contains substring)
mask_auth_names = df_trace["trace_name"].str.contains("ts-auth-service", case=False, na=False)
auth_trace_names = df_trace.loc[mask_auth_names, "trace_name"].unique()

# 1) Compute global p95 for each trace_name (full series)
p95_full = (
    df_trace.groupby("trace_name", dropna=False)["value"]
    .quantile(0.95)
    .reset_index(name="p95")
)
# Keep only auth-related trace_names p95s
p95_auth = p95_full[p95_full["trace_name"].isin(auth_trace_names)].copy()

# 2) Filter trace rows in the incident window and for auth-related trace_names
df_auth_window = df_trace[mask_auth_names].copy()
df_auth_window = df_auth_window[(df_auth_window["timestamp"] >= start) & (df_auth_window["timestamp"] <= end)]

# Merge global p95 into the windowed rows
df_auth_window = df_auth_window.merge(p95_auth, on="trace_name", how="left")

# 2) Mark rows exceeding p95 (value > p95) and compute exceedance_ratio
# Handle p95 == 0 by setting exceedance_ratio = inf when value > 0
def compute_exceed_ratio(row):
    p95 = row["p95"]
    val = row["value"]
    if pd.isna(p95):
        return np.nan
    if p95 == 0:
        return np.inf if val > 0 else np.nan
    return val / p95

df_auth_window["exceed"] = df_auth_window["value"] > df_auth_window["p95"]
df_auth_window["exceedance_ratio"] = df_auth_window.apply(compute_exceed_ratio, axis=1)

# Select only anomalies (value > p95)
anomalies_auth = df_auth_window[df_auth_window["exceed"]].copy()

# Format timestamp to ISO UTC string
if not anomalies_auth.empty:
    anomalies_auth["timestamp_iso"] = anomalies_auth["timestamp"].dt.tz_convert("UTC").dt.strftime("%Y-%m-%dT%H:%M:%SZ")
else:
    anomalies_auth["timestamp_iso"] = pd.Series(dtype="object")

# 3) Prepare top 20 rows sorted by exceedance_ratio desc with requested columns
anomalies_top20 = (
    anomalies_auth.sort_values("exceedance_ratio", ascending=False)
    .loc[:, ["trace_name", "timestamp_iso", "value", "p95", "exceedance_ratio"]]
    .rename(columns={"timestamp_iso": "timestamp"})
    .head(20)
    .reset_index(drop=True)
)

# 4) Small summary: earliest anomaly timestamp, total auth-trace anomalies, max exceedance_ratio
if anomalies_auth.empty:
    summary = {
        "earliest_anomaly_timestamp": None,
        "total_auth_trace_anomalies": 0,
        "max_exceedance_ratio": None,
    }
else:
    earliest = anomalies_auth["timestamp"].min().tz_convert("UTC").strftime("%Y-%m-%dT%H:%M:%SZ")
    total_count = int(anomalies_auth.shape[0])
    max_exc = float(anomalies_auth["exceedance_ratio"].replace([np.inf, -np.inf], np.nan).max()) \
        if np.isfinite(anomalies_auth["exceedance_ratio"].replace([np.inf, -np.inf], np.nan).max()) else np.inf
    summary = {
        "earliest_anomaly_timestamp": earliest,
        "total_auth_trace_anomalies": total_count,
        "max_exceedance_ratio": max_exc,
    }

summary_df = pd.DataFrame([summary])

# Return compact results
anomalies_top20, summary_df

```
Out[3]:
```
Summary (plain English):

- Global p95 values (computed on the full trace.csv series) for the auth-related trace names seen in the incident window:
  - trace.from_ts-auth-service.duration_p95 — p95 = 0.003220
  - trace.to_ts-auth-service.duration_p95 — p95 = 0.836731
  - trace.from_ts-auth-service.duration_mean — p95 = 0.003943
  - trace.to_ts-auth-service.duration_mean — p95 = 0.353188
  - trace.from_ts-auth-service.row_count — p95 = 359.2
  - trace.to_ts-auth-service.row_count — p95 = 348.0

- Anomalies in the incident window (value > global p95):
  - Total auth-trace anomalies found in the window: 14.
  - Earliest anomaly timestamp: 2024-01-25T03:18:00Z.
  - Maximum exceedance ratio: ~20.63×.

- Top anomaly (most severe):
  - trace.from_ts-auth-service.duration_p95 at 2024-01-25T03:18:00Z — value 0.066443 vs p95 0.003220 → exceedance_ratio ≈ 20.63.

- Other notable auth-trace spikes in the window:
  - trace.to_ts-auth-service.duration_p95 at 2024-01-25T03:18:00Z — exceedance_ratio ≈ 1.76.
  - Several duration_mean and row_count rows are modestly above their p95 (ratios ~1.03–1.60).

Recommendation:
- Investigate ts-auth-service latency spike around 03:17–03:18 UTC (the large duration_p95 spike) as highest-priority root cause candidate; also review increased row_count and elevated duration_mean/duration_p95 around the same times for broader impact.

The original code execution output of IPython Kernel is also provided below for reference:

(                                  trace_name             timestamp       value         p95  exceedance_ratio
0    trace.from_ts-auth-service.duration_p95  2024-01-25T03:18:00Z    0.066443    0.003220         20.633932
1      trace.to_ts-auth-service.duration_p95  2024-01-25T03:18:00Z    1.475890    0.836731          1.763876
2   trace.from_ts-auth-service.duration_mean  2024-01-25T03:18:00Z    0.006296    0.003943          1.596675
3     trace.to_ts-auth-service.duration_mean  2024-01-25T03:18:00Z    0.530734    0.353188          1.502694
4     trace.to_ts-auth-service.duration_mean  2024-01-25T03:19:00Z    0.390218    0.353188          1.104845
5         trace.to_ts-auth-service.row_count  2024-01-25T03:22:00Z  378.000000  348.000000          1.086207
6      trace.to_ts-auth-service.duration_p95  2024-01-25T03:29:00Z    0.893919    0.836731          1.068346
7       trace.from_ts-auth-service.row_count  2024-01-25T03:22:00Z  378.000000  359.200000          1.052339
8         trace.to_ts-auth-service.row_count  2024-01-25T03:26:00Z  362.000000  348.000000          1.040230
9      trace.to_ts-auth-service.duration_p95  2024-01-25T03:19:00Z    0.865068    0.836731          1.033866
10    trace.to_ts-auth-service.duration_mean  2024-01-25T03:23:00Z    0.363933    0.353188          1.030423
11  trace.from_ts-auth-service.duration_mean  2024-01-25T03:19:00Z    0.004040    0.003943          1.024630
12   trace.from_ts-auth-service.duration_p95  2024-01-25T03:20:00Z    0.003255    0.003220          1.011006
13      trace.from_ts-auth-service.row_count  2024-01-25T03:26:00Z  362.000000  359.200000          1.007795,   earliest_anomaly_timestamp  total_auth_trace_anomalies  max_exceedance_ratio
0       2024-01-25T03:18:00Z                          14             20.633932)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Reuse existing DataFrames if present; otherwise load them
try:
    df_metric
    df_trace
except NameError:
    df_metric = pd.read_csv("metric.csv")
    df_trace = pd.read_csv("trace.csv")

# Ensure timestamps are parsed as UTC datetimes (re-apply safely)
if "timestamp" in df_metric.columns:
    df_metric["timestamp"] = pd.to_datetime(df_metric["timestamp"], unit="s", utc=True)
if "timestamp" in df_trace.columns:
    df_trace["timestamp"] = pd.to_datetime(df_trace["timestamp"], unit="s", utc=True)

# Part 1: ts-auth-service metric anomalies within incident window
component = "ts-auth-service"
start = pd.to_datetime("2024-01-25 03:14:30", utc=True)
end = pd.to_datetime("2024-01-25 03:44:30", utc=True)

# Filter metric rows for the component to compute per-kpi global p95 (full series for that component)
df_metric_auth = df_metric[df_metric["cmdb_id"] == component].copy()

if df_metric_auth.empty:
    # No data for ts-auth-service in metric.csv
    metrics_anomalies_top20 = pd.DataFrame(columns=["kpi_name","timestamp","value","p95","exceedance_ratio"])
    metric_summary_df = pd.DataFrame([{
        "component": component,
        "total_metric_anomalies": "no data",
        "earliest_metric_anomaly_timestamp": "no data",
        "max_metric_exceedance_ratio": "no data"
    }])
else:
    # Compute global p95 per kpi_name for ts-auth-service using full available series (rule 9)
    metric_p95 = (
        df_metric_auth.groupby("kpi_name", dropna=False)["value"]
        .quantile(0.95)
        .reset_index(name="p95")
    )
    # Also capture p95 as numeric and avoid NaNs
    # Filter metric.csv for the incident window and the component
    metric_window = df_metric_auth[(df_metric_auth["timestamp"] >= start) & (df_metric_auth["timestamp"] <= end)].copy()
    # Merge global p95 per kpi_name
    metric_window = metric_window.merge(metric_p95, on="kpi_name", how="left")
    # Keep only rows where p95 is present
    metric_window = metric_window.dropna(subset=["p95"])
    # Determine anomalies: value > p95 (high-sided as requested)
    def compute_exc_ratio(row):
        p95 = row["p95"]
        val = row["value"]
        if pd.isna(p95):
            return np.nan
        if p95 == 0:
            return np.inf if val > 0 else np.nan
        return val / p95
    if not metric_window.empty:
        metric_window["is_anomaly"] = metric_window["value"] > metric_window["p95"]
        metric_window["exceedance_ratio"] = metric_window.apply(compute_exc_ratio, axis=1)
        anomalies_metric = metric_window[metric_window["is_anomaly"]].copy()
        # Format timestamp to ISO UTC
        anomalies_metric["timestamp_iso"] = anomalies_metric["timestamp"].dt.tz_convert("UTC").dt.strftime("%Y-%m-%dT%H:%M:%SZ")
        # Prepare final columns and sort by exceedance_ratio desc
        metrics_anomalies_top20 = (
            anomalies_metric.loc[:, ["kpi_name", "timestamp_iso", "value", "p95", "exceedance_ratio"]]
            .rename(columns={"timestamp_iso": "timestamp"})
            .sort_values("exceedance_ratio", ascending=False)
            .head(20)
            .reset_index(drop=True)
        )
        # Summary
        if anomalies_metric.empty:
            metric_summary_df = pd.DataFrame([{
                "component": component,
                "total_metric_anomalies": 0,
                "earliest_metric_anomaly_timestamp": None,
                "max_metric_exceedance_ratio": np.nan
            }])
        else:
            earliest = anomalies_metric["timestamp"].min().tz_convert("UTC").strftime("%Y-%m-%dT%H:%M:%SZ")
            total = int(anomalies_metric.shape[0])
            max_exc = anomalies_metric["exceedance_ratio"].replace([np.inf, -np.inf], np.nan).max()
            # Keep numeric or inf
            if np.isfinite(max_exc):
                max_exc_val = float(max_exc)
            else:
                max_exc_val = np.inf
            metric_summary_df = pd.DataFrame([{
                "component": component,
                "total_metric_anomalies": total,
                "earliest_metric_anomaly_timestamp": earliest,
                "max_metric_exceedance_ratio": max_exc_val
            }])
    else:
        # No rows in window or no p95 available
        metrics_anomalies_top20 = pd.DataFrame(columns=["kpi_name","timestamp","value","p95","exceedance_ratio"])
        metric_summary_df = pd.DataFrame([{
            "component": component,
            "total_metric_anomalies": 0,
            "earliest_metric_anomaly_timestamp": None,
            "max_metric_exceedance_ratio": np.nan
        }])

# Part 2: Trace context around 03:17, 03:18, 03:19 for candidate services
candidates = ["ts-auth-service", "ts-order-service", "ts-route-service", "ts-train-service", "ts-travel-service"]
# Timestamps of interest (exact)
ts_list = [
    pd.to_datetime("2024-01-25T03:17:00Z", utc=True),
    pd.to_datetime("2024-01-25T03:18:00Z", utc=True),
    pd.to_datetime("2024-01-25T03:19:00Z", utc=True),
]
# Compute global p95 per trace_name across full trace series (rule 9)
trace_p95_full = (
    df_trace.groupby("trace_name", dropna=False)["value"]
    .quantile(0.95)
    .reset_index(name="p95")
)

# Filter df_trace to the three timestamps and trace_names that mention any candidate services
# First build pattern to match any candidate substring
pattern = "(" + "|".join([p.replace("-", r"\-") for p in candidates]) + ")"
# Filter by trace_name containing any candidate service (case-insensitive)
mask_candidate_traces = df_trace["trace_name"].str.contains("|".join(candidates), case=False, na=False)
# Filter by timestamp equals one of the ts_list
mask_timestamp = df_trace["timestamp"].isin(ts_list)
trace_context = df_trace[mask_candidate_traces & mask_timestamp].copy()

# If empty, prepare empty DataFrame with requested columns
if trace_context.empty:
    traces_context_top100 = pd.DataFrame(columns=["timestamp","component","trace_name","value","p95","exceedance_ratio","is_anomaly"])
else:
    # Merge global p95 for each trace_name
    trace_context = trace_context.merge(trace_p95_full, on="trace_name", how="left")
    # Compute exceedance_ratio and is_anomaly (value > p95)
    def compute_exc_trace(row):
        p95 = row["p95"]
        val = row["value"]
        if pd.isna(p95):
            return (np.nan, False)
        if p95 == 0:
            exc = np.inf if val > 0 else np.nan
            is_anom = val > 0
            return (exc, is_anom)
        else:
            return (val / p95, val > p95)
    results = trace_context.apply(lambda r: compute_exc_trace(r), axis=1)
    trace_context[["exceedance_ratio", "is_anomaly"]] = pd.DataFrame(results.tolist(), index=trace_context.index)
    # Format timestamp ISO
    trace_context["timestamp_iso"] = trace_context["timestamp"].dt.tz_convert("UTC").dt.strftime("%Y-%m-%dT%H:%M:%SZ")
    # Prepare final columns
    traces_context_top100 = (
        trace_context.loc[:, ["timestamp_iso", "cmdb_id", "trace_name", "value", "p95", "exceedance_ratio", "is_anomaly"]]
        .rename(columns={"timestamp_iso": "timestamp", "cmdb_id": "component"})
        .sort_values(["timestamp", "exceedance_ratio"], ascending=[True, False])
        .head(100)
        .reset_index(drop=True)
    )

# Final outputs (compact)
metrics_anomalies_top20, metric_summary_df, traces_context_top100

```
Out[4]:
```
Summary (plain English)

Part 1 — ts-auth-service metrics (incident window 2024-01-25 03:14:30 → 03:44:30 UTC)
- Global p95s were computed from the full metric series for ts-auth-service, and metric rows in the incident window exceeding those p95s were reported.
- Total metric anomalies found: 14.
- Earliest metric anomaly: 2024-01-25T03:17:00Z.
- Maximum exceedance ratio (value / p95): 1.670235 (the top metric anomaly is latency-90 at 2024-01-25T03:19:00Z, value ≈ 1.4558 vs p95 ≈ 0.8716).
- Other metric anomalies are modest spikes (cpu, diskio, socket, latency-50, mem, workload) with exceedance ratios mostly near 1.01–1.08.

Interpretation: ts-auth-service shows multiple metric-level spikes during the window, the largest being latency-90 (~1.67× p95). These are noticeable but not as extreme as the severe trace-level auth spike seen earlier.

Part 2 — Trace context at 03:17:00Z, 03:18:00Z, 03:19:00Z for candidate services
- We extracted trace rows at the three timestamps for trace_names mentioning any candidate service, merged full-series p95 for each trace_name, computed exceedance_ratio and flagged whether value > p95.
- Output contains up to 100 context rows (sorted by timestamp, then exceedance_ratio). Examples:
  - Several small but true anomalies at 03:17:00Z for route/travel services (exceedance ratios ~1.01–1.08).
  - For ts-auth-service specifically, the major trace anomalies occur at 03:18:00Z (previously identified): e.g., trace.from_ts-auth-service.duration_p95 at 03:18:00Z is a very large spike (value 0.066443 vs p95 0.003220 → ~20.63×). That severe auth trace spike is present in the broader context window and is the dominant trace anomaly.
  - Many other trace rows at 03:17–03:19 are below their p95 (not anomalies) or show modest exceedances.

Compact conclusions / next steps
- The top priority is the ts-auth-service trace latency spike at ~03:18:00Z (huge exceedance ~20× p95). This aligns with metric-level latency increases nearby (03:17–03:19) though metric exceedances are smaller (max ~1.67×).
- Trace context shows modest spikes across other services at 03:17–03:19 (route, order, travel), suggesting broader elevated latencies/traffic, but none match the severity of the auth trace spike.
- Investigate ts-auth-service latency at ~03:18 UTC first (trace-level spike), then review related services (ts-travel, ts-order, ts-route) for correlated increases.

The original code execution output of IPython Kernel is also provided below for reference:

(      kpi_name             timestamp         value           p95  exceedance_ratio
0   latency-90  2024-01-25T03:19:00Z  1.455803e+00  8.716158e-01          1.670235
1   latency-50  2024-01-25T03:18:00Z  3.035815e-01  2.793133e-01          1.086885
2       socket  2024-01-25T03:19:00Z  1.648333e+01  1.534667e+01          1.074066
3          cpu  2024-01-25T03:20:00Z  1.726280e+01  1.624511e+01          1.062645
4       diskio  2024-01-25T03:18:00Z  5.752636e+04  5.570824e+04          1.032636
5   latency-90  2024-01-25T03:18:00Z  8.879972e-01  8.716158e-01          1.018794
6     workload  2024-01-25T03:17:00Z  3.048767e+00  3.020570e+00          1.009335
7          mem  2024-01-25T03:20:00Z  2.517388e+08  2.494849e+08          1.009034
8       socket  2024-01-25T03:18:00Z  1.543333e+01  1.534667e+01          1.005647
9          cpu  2024-01-25T03:17:00Z  1.632706e+01  1.624511e+01          1.005044
10    workload  2024-01-25T03:22:00Z  3.032117e+00  3.020570e+00          1.003823
11         mem  2024-01-25T03:21:00Z  2.498085e+08  2.494849e+08          1.001297
12      diskio  2024-01-25T03:34:00Z  5.573043e+04  5.570824e+04          1.000398
13  latency-50  2024-01-25T03:24:00Z  2.793744e-01  2.793133e-01          1.000219,          component  total_metric_anomalies earliest_metric_anomaly_timestamp  max_metric_exceedance_ratio
0  ts-auth-service                      14              2024-01-25T03:17:00Z                     1.670235,                timestamp                  component                                 trace_name     value          p95  exceedance_ratio  is_anomaly
0   2024-01-25T03:17:00Z         ts-travel2-service     trace.to_ts-route-service.duration_p95  0.005511     0.005120          1.076555        True
1   2024-01-25T03:17:00Z           ts-basic-service    trace.to_ts-route-service.duration_mean  0.003901     0.003859          1.010886        True
2   2024-01-25T03:17:00Z         ts-travel2-service    trace.to_ts-route-service.duration_mean  0.003890     0.003859          1.008045        True
3   2024-01-25T03:17:00Z           ts-route-service  trace.from_ts-route-service.duration_mean  0.001328     0.001339          0.991847       False
4   2024-01-25T03:17:00Z           ts-basic-service     trace.to_ts-route-service.duration_p95  0.005040     0.005120          0.984457       False
..                   ...                        ...                                        ...       ...          ...               ...         ...
95  2024-01-25T03:17:00Z            ts-auth-service      trace.to_ts-auth-service.duration_p95  0.002082     0.836731          0.002488       False
96  2024-01-25T03:17:00Z  ts-inside-payment-service        trace.to_ts-order-service.row_count  1.000000   480.000000          0.002083       False
97  2024-01-25T03:17:00Z        ts-preserve-service       trace.to_ts-travel-service.row_count  3.000000  1628.750000          0.001842       False
98  2024-01-25T03:17:00Z            ts-seat-service   trace.to_ts-travel-service.duration_mean  0.015473    11.202487          0.001381       False
99  2024-01-25T03:17:00Z          ts-travel-service   trace.to_ts-travel-service.duration_mean  0.011911    11.202487          0.001063       False

[100 rows x 7 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np

# Reuse existing DataFrames if present; otherwise read files
try:
    df_log
    df_error
except NameError:
    df_log = pd.read_csv("log.csv")
    df_error = pd.read_csv("error_logs.csv")

# Ensure timestamps parsed as UTC datetimes
if "timestamp" in df_log.columns:
    df_log["timestamp"] = pd.to_datetime(df_log["timestamp"], unit="s", utc=True)
if "timestamp" in df_error.columns:
    df_error["timestamp"] = pd.to_datetime(df_error["timestamp"], unit="s", utc=True)

# Incident window
start = pd.to_datetime("2024-01-25 03:14:30", utc=True)
end = pd.to_datetime("2024-01-25 03:44:30", utc=True)

# Part 1: ts-auth-service log anomalies
component = "ts-auth-service"
df_log_auth_full = df_log[df_log["cmdb_id"] == component].copy()

if df_log_auth_full.empty:
    # No log data for component at all
    auth_log_anomalies_top20 = pd.DataFrame(columns=["log_name","timestamp","value","p95","exceedance_ratio"])
else:
    # Compute global p95 per log_name using full series for ts-auth-service (do NOT filter first)
    log_p95 = (
        df_log_auth_full.groupby("log_name", dropna=False)["value"]
        .quantile(0.95)
        .reset_index(name="p95")
    )
    # Filter to incident window
    df_log_auth_window = df_log_auth_full[(df_log_auth_full["timestamp"] >= start) & (df_log_auth_full["timestamp"] <= end)].copy()
    if df_log_auth_window.empty:
        auth_log_anomalies_top20 = pd.DataFrame(columns=["log_name","timestamp","value","p95","exceedance_ratio"])
    else:
        # Merge p95 into window rows
        df_log_auth_window = df_log_auth_window.merge(log_p95, on="log_name", how="left")
        # Drop rows without p95 (shouldn't happen)
        df_log_auth_window = df_log_auth_window.dropna(subset=["p95"])
        # Determine anomalies: value > p95
        def compute_exc(row):
            p = row["p95"]
            v = row["value"]
            if pd.isna(p):
                return np.nan
            if p == 0:
                return np.inf if v > 0 else np.nan
            return v / p
        df_log_auth_window["is_anomaly"] = df_log_auth_window["value"] > df_log_auth_window["p95"]
        df_log_auth_window["exceedance_ratio"] = df_log_auth_window.apply(compute_exc, axis=1)
        anomalies = df_log_auth_window[df_log_auth_window["is_anomaly"]].copy()
        if anomalies.empty:
            auth_log_anomalies_top20 = pd.DataFrame(columns=["log_name","timestamp","value","p95","exceedance_ratio"])
        else:
            anomalies["timestamp_iso"] = anomalies["timestamp"].dt.tz_convert("UTC").dt.strftime("%Y-%m-%dT%H:%M:%SZ")
            auth_log_anomalies_top20 = (
                anomalies.loc[:, ["log_name", "timestamp_iso", "value", "p95", "exceedance_ratio"]]
                .rename(columns={"timestamp_iso": "timestamp"})
                .sort_values("exceedance_ratio", ascending=False)
                .head(20)
                .reset_index(drop=True)
            )

# Part 2: error_logs for ts-auth-service in incident window
df_error_auth_window = df_error[(df_error["cmdb_id"] == component) & (df_error["timestamp"] >= start) & (df_error["timestamp"] <= end)].copy()

if df_error_auth_window.empty:
    auth_error_logs_result = pd.DataFrame([{
        "timestamp": "",
        "cmdb_id": component,
        "message": "no error_logs for ts-auth-service in window"
    }], columns=["timestamp", "cmdb_id", "message"])
else:
    df_error_auth_window["timestamp_iso"] = df_error_auth_window["timestamp"].dt.tz_convert("UTC").dt.strftime("%Y-%m-%dT%H:%M:%SZ")
    auth_error_logs_result = df_error_auth_window.loc[:, ["timestamp_iso", "cmdb_id", "message"]].rename(columns={"timestamp_iso": "timestamp"})
    auth_error_logs_result = auth_error_logs_result.head(20).reset_index(drop=True)

# Return compact outputs
auth_log_anomalies_top20, auth_error_logs_result

```
Out[5]:
```
Summary (plain English):

Part 1 — ts-auth-service log anomalies (incident window 2024-01-25 03:14:30 → 03:44:30 UTC)
- Global p95 for the relevant log KPI (computed on the full log.csv for ts-auth-service):
  - log.row_count p95 = 359.2
- Log rows in the incident window with value > p95 (top results):
  1. log.row_count at 2024-01-25T03:22:00Z — value 378, p95 359.2, exceedance_ratio ≈ 1.0523
  2. log.row_count at 2024-01-25T03:26:00Z — value 362, p95 359.2, exceedance_ratio ≈ 1.0078
- Summary for ts-auth-service logs:
  - total log anomalies found: 2
  - earliest log anomaly: 2024-01-25T03:22:00Z
  - max exceedance_ratio: ≈ 1.0523

Part 2 — error_logs for ts-auth-service in the window
- No error_logs rows found for ts-auth-service within the incident window.
  - (Explicit result: "no error_logs for ts-auth-service in window")

The original code execution output of IPython Kernel is also provided below for reference:

(        log_name             timestamp  value    p95  exceedance_ratio
0  log.row_count  2024-01-25T03:22:00Z    378  359.2          1.052339
1  log.row_count  2024-01-25T03:26:00Z    362  359.2          1.007795,   timestamp          cmdb_id                                      message
0            ts-auth-service  no error_logs for ts-auth-service in window)```
```